# EE 123 Lab 3 — Sampling & Compression: Audio + Image

### Written by Tingle Li and John Wang, Spring 2026

In this lab you will connect **sampling**, **resampling**, and **quantization** (ADC/DAC view) with **transform compression** (a JPEG-like pipeline).
You will work with both **audio** (listen + spectrograms) and **images** (visualize + frequency/transform coefficients).

**Prerequisites:** Lab 2 (spectral analysis, 2D DFT, STFT).


In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import scipy.fft as fft
import librosa
import librosa.display
from skimage import data as skdata
from skimage import color, transform, util
from PIL import Image
from io import BytesIO
from collections import Counter
import heapq
import time
import math

from IPython.display import Audio, display

np.random.seed(0)
plt.rcParams['figure.figsize'] = (10, 4)


In [ ]:
# Assets: images
img_color = skdata.astronaut()                       # RGB uint8
img_gray = util.img_as_float(skdata.camera())        # grayscale float in [0,1]
img_color2 = skdata.coffee()                         # RGB uint8 (for R-D comparison)

# A high-frequency checkerboard (aliasing stress-test)
H, W = img_gray.shape
def make_checkerboard(h=512, w=512, block=8):
    yy, xx = np.indices((h, w))
    return ((yy // block + xx // block) % 2).astype(float)

checker = make_checkerboard(H, W, block=4)

# Assets: audio — orchestra recording at 44.1 kHz (included as WAV)
x_audio, fs_audio = librosa.load('assets/orchestra.wav', sr=None, mono=True)

print("Image shapes:", img_color.shape, img_gray.shape, img_color2.shape)
print("Audio:", x_audio.shape, "fs =", fs_audio)

# -----------------------
# Helper functions (provided)
# -----------------------
def plot_mag_spectrum(x, fs, n_fft=None, title=None, xlim=None):
    """Plot a one-sided magnitude spectrum in dB."""
    if n_fft is None:
        n_fft = int(2**np.ceil(np.log2(len(x))))
    X = np.fft.rfft(x, n=n_fft)
    f = np.fft.rfftfreq(n_fft, d=1/fs)
    mag_db = 20*np.log10(np.maximum(np.abs(X), 1e-12))
    plt.figure(figsize=(10,3))
    plt.plot(f, mag_db)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude (dB)")
    if title: plt.title(title)
    if xlim: plt.xlim(xlim)
    plt.grid(True, alpha=0.3)
    plt.show()
    return f, mag_db

def show_fft2_logmag(img, title=None):
    """Show log-magnitude of 2D FFT (centered)."""
    F = np.fft.fftshift(np.fft.fft2(img))
    mag = np.log1p(np.abs(F))
    plt.figure(figsize=(4,4))
    plt.imshow(mag, cmap='gray')
    plt.axis('off')
    if title: plt.title(title)
    plt.show()

def psnr(x, y, data_range=1.0):
    mse = np.mean((x - y)**2)
    if mse == 0:
        return np.inf
    return 10*np.log10((data_range**2) / mse)

def dct2(block):
    return fft.dct(fft.dct(block, axis=0, norm='ortho'), axis=1, norm='ortho')

def idct2(block):
    return fft.idct(fft.idct(block, axis=0, norm='ortho'), axis=1, norm='ortho')

def pad_to_block(img, block=8):
    """Pad 2D image so H and W are multiples of block."""
    H, W = img.shape
    Hp = int(np.ceil(H / block) * block)
    Wp = int(np.ceil(W / block) * block)
    pad_h = Hp - H
    pad_w = Wp - W
    return np.pad(img, ((0, pad_h), (0, pad_w)), mode='edge'), (H, W)

def block_view(img, block=8):
    """Return a view of image as (nB_H, nB_W, block, block)."""
    H, W = img.shape
    assert H % block == 0 and W % block == 0
    return img.reshape(H//block, block, W//block, block).swapaxes(1,2)

def unblock_view(blocks):
    """Inverse of block_view."""
    nBh, nBw, b1, b2 = blocks.shape
    return blocks.swapaxes(1,2).reshape(nBh*b1, nBw*b2)


In [ ]:
# Extra helper: spectrogram
def show_spectrogram(x, fs, n_fft=1024, hop_length=256, title=None, fmax=None):
    S = librosa.stft(x, n_fft=n_fft, hop_length=hop_length, window='hann')
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    plt.figure(figsize=(10,4))
    librosa.display.specshow(S_db, sr=fs, hop_length=hop_length, x_axis='time', y_axis='hz')
    if fmax is not None:
        plt.ylim(0, fmax)
    plt.colorbar(format='%+2.0f dB')
    if title: plt.title(title)
    plt.tight_layout()
    plt.show()


---

# Part 0: Warm-up — Raw data size

This part is short, but sets the stage: sampling rate / bit depth / resolution determine **raw** data size.

## Task 0.1: Compute raw bitrate / raw image size

Fill in the missing computations below, and report results in a small markdown table.


In [ ]:
# Task 0.1

# Audio: assume mono PCM, duration = 5 seconds
duration_sec = 5
fs = fs_audio
bits_per_sample = 16
channels = 1

# TODO: raw_audio_bits = ?
raw_audio_bits = # TODO
raw_audio_kbits_per_sec = # TODO
raw_audio_kbytes_total = # TODO

# Image: assume RGB uint8 (24 bpp) for img_color
Hc, Wc, Cc = img_color.shape
bits_per_pixel = 24

# TODO: raw_image_bits = ?
raw_image_bits = # TODO
raw_image_kbytes_total = # TODO

print("Audio raw kbps:", raw_audio_kbits_per_sec)
print("Audio raw total (KB):", raw_audio_kbytes_total)
print("Image raw total (KB):", raw_image_kbytes_total)


**Question:**  
1. In one sentence: what are the two big “levers” that compression algorithms use to reduce size?  
2. If you only reduce sampling rate / resolution, what kind of quality loss do you expect compared to transform compression (JPEG/MP3)?  


---

# Part 1: Sampling & Aliasing (audio + image)

In Lab 2 you analyzed spectra. Here you will use that skill to understand **what goes wrong** when sampling is too low.

## Task 1.1: Audio aliasing by naive downsampling

Aliasing is one of the most important phenomena in DSP. In this task, you will **hear** it directly: downsample a orchestra recording without an anti-aliasing filter, and listen to the distortion.


In [ ]:
# Task 1.1 — Audio aliasing demo

x = x_audio.copy()
Fs0 = fs_audio

print(f"Original: Fs = {Fs0} Hz, duration = {len(x)/Fs0:.2f} s")
display(Audio(x, rate=Fs0))

# TODO: Plot the magnitude spectrum of the original signal
# TODO

# Downsample factors to test
Ms = [2, 3, 4]
for M in Ms:
    Fs_ds = Fs0 // M

    # TODO: Downsample x by factor M without any filtering
    x_ds = # TODO

    print(f"\n--- M={M}, Fs_ds={Fs_ds} Hz ---")
    display(Audio(x_ds, rate=Fs_ds))

    # TODO: Plot the magnitude spectrum of the downsampled signal
    # TODO


**Question:**  
Compare the original audio spectrum with the M=4 downsampled spectrum. Point out which spectral peaks are **new** (i.e., produced by aliasing — mark them on your plot). Use the Nyquist theorem to explain why aliasing gets worse as M increases.


## Task 1.2: Image aliasing — moiré, false color, and frequency-domain evidence

You will compare:
- naive subsampling (`img[::M, ::M]`)
- anti-aliased resize (`skimage.transform.resize(..., anti_aliasing=True)`)

Use both a **high-frequency pattern** (checkerboard) and a **real color image** (astronaut RGB) so you can observe chromatic aliasing (false colors).


In [ ]:
# Task 1.2 — Image aliasing (color + checkerboard)

imgs = {
    "checker": checker,
    "astronaut": util.img_as_float(img_color),
}

Ms = [2, 3, 4]

for name, img in imgs.items():
    is_color = img.ndim == 3
    img_luma = color.rgb2gray(img) if is_color else img
    cmap = None if is_color else 'gray'

    # --- Row 1: original image + its FFT2 ---
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img, cmap=cmap); axes[0].set_title(f"{name}: Original"); axes[0].axis('off')
    F = np.fft.fftshift(np.fft.fft2(img_luma))
    axes[1].imshow(np.log1p(np.abs(F)), cmap='gray'); axes[1].set_title(f"{name}: FFT2 (original)"); axes[1].axis('off')
    plt.tight_layout(); plt.show()

    # --- For each M: one row with [naive, AA, naive FFT2, AA FFT2] ---
    for M in Ms:
        if is_color:
            img_naive = img[::M, ::M, :]
        else:
            img_naive = img[::M, ::M]

        # TODO: Anti-aliased resize to the same output size as img_naive
        out_shape = img_naive.shape[:2] if is_color else img_naive.shape
        img_aa = # TODO

        naive_luma = color.rgb2gray(img_naive) if is_color else img_naive
        aa_luma = color.rgb2gray(img_aa) if is_color else img_aa

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(img_naive, cmap=cmap); axes[0].set_title(f"Naive M={M}"); axes[0].axis('off')
        axes[1].imshow(img_aa, cmap=cmap); axes[1].set_title(f"AA resize M={M}"); axes[1].axis('off')
        F_n = np.fft.fftshift(np.fft.fft2(naive_luma))
        axes[2].imshow(np.log1p(np.abs(F_n)), cmap='gray'); axes[2].set_title(f"Naive FFT2"); axes[2].axis('off')
        F_a = np.fft.fftshift(np.fft.fft2(aa_luma))
        axes[3].imshow(np.log1p(np.abs(F_a)), cmap='gray'); axes[3].set_title(f"AA FFT2"); axes[3].axis('off')
        plt.suptitle(f"{name}, M={M}", fontsize=13)
        plt.tight_layout(); plt.show()


**Question:**  
Use the FFT2 visualizations to explain how the moiré pattern on the checkerboard is produced (frequency overlap / folding). On the color astronaut, do you see **false colors** that don't exist in the original? Explain why independently downsampling the R, G, B channels produces false color, and why anti-aliased resize reduces these artifacts.


---

# Part 2: Anti-aliasing & Resampling

Now you will fix the aliasing problem using **anti-aliasing filters**, and then tackle a real resampling problem (44.1 kHz → 48 kHz).

## Task 2.1: Design an anti-aliasing FIR for audio decimation


In [ ]:
# Task 2.1 — Anti-aliasing FIR + decimation

# Use a real audio clip (short)
x = x_audio
Fs0 = fs_audio

M = 3
Fs_ds = Fs0 / M

# Naive downsample (baseline)
x_naive = x[::M]

print("Original:")
display(Audio(x, rate=Fs0))
print("Naive downsample:")
display(Audio(x_naive, rate=int(Fs_ds)))

# TODO: Design a low-pass FIR anti-aliasing filter.
# Think about what cutoff frequency prevents aliasing after decimation by M.
numtaps_list = [31, 101, 401]
cutoff_list = [0.45*(Fs0/(2*M)), 0.9*(Fs0/(2*M)), 1.1*(Fs0/(2*M))]  # you may edit

results = []  # store (numtaps, cutoff_hz, metric, ...)

for numtaps in numtaps_list:
    for cutoff_hz in cutoff_list:
        # TODO: Design filter
        h = # TODO

        # filter then decimate
        x_f = signal.lfilter(h, [1.0], x)
        x_ds = x_f[::M]

        # TODO: Define and compute a metric that measures how much aliasing is present
        metric = # TODO

        results.append((numtaps, cutoff_hz, metric))

        print(f"numtaps={numtaps}, cutoff={cutoff_hz:.1f} Hz, metric={metric}")
        # Optional: listen to a couple configs (don't listen to all)
        # display(Audio(x_ds, rate=int(Fs_ds)))

# TODO: Pick your best configuration and justify your choice
best = # TODO
print("Best config:", best)


**Question:**  
1. Explain how you chose your cutoff and numtaps (2–4 sentences).  
2. What happens if cutoff is too high? too low?  
3. What happens if numtaps is too small? Use at least one plot/spectrogram as evidence.


## Task 2.2: Rational resampling — 44.1 kHz → 8 kHz

Resampling between rates that are not integer multiples requires a **rational resampler**: upsample by L, low-pass filter, then downsample by M, where L/M = Fs_out/Fs_in (reduced fraction).

Here you will downsample a 44.1 kHz orchestra recording to 8 kHz (telephone quality) — a dramatic change you can clearly hear.

You will:
1. Compute the reduced fraction L/M.  
2. Implement a **naive** rational resampler: upsample by L → low-pass → downsample by M.  
3. Compare against `signal.resample_poly` (reference).


In [ ]:
# Task 2.2 — 44.1 kHz → 8 kHz rational resampling

Fs_in = 44100
Fs_out = 8000

x_in = x_audio[:int(3 * fs_audio)].copy()  # use 3 seconds to keep naive resampler fast

print(f"Input: {len(x_in)} samples at {Fs_in} Hz ({len(x_in)/Fs_in:.1f}s)")
display(Audio(x_in, rate=Fs_in))

# --- Step 1: find L/M ---
# TODO: Compute L and M such that Fs_out/Fs_in = L/M (reduced fraction)
L = # TODO
M = # TODO
print("L/M =", L, "/", M, "=", L/M)

# --- Step 2: naive resampler ---
# TODO: Upsample x_in by factor L
x_up = # TODO

# TODO: Design an interpolation low-pass FIR for the upsampled signal.
# What cutoff frequency removes the spectral images introduced by upsampling?
numtaps = 401
cutoff_hz = # TODO
h = # TODO

# Filter
x_f = signal.lfilter(h, [1.0], x_up)

# TODO: Downsample by factor M
x_naive = # TODO

# --- Step 3: reference method ---
x_ref = signal.resample_poly(x_in, up=L, down=M)

print(f"Output lengths: naive={len(x_naive)}, ref={len(x_ref)}")

# --- Listen and compare ---
print("\nNaive resampler:")
display(Audio(x_naive, rate=Fs_out))
print("Reference (resample_poly):")
display(Audio(x_ref, rate=Fs_out))

# TODO: Compute a relative error metric between your naive result and the reference
err = # TODO
print("Relative error:", err)

# TODO: Plot spectrograms of both results for visual comparison


**Question:**  
What L/M did you obtain? Compare your naive resampler against `resample_poly`: where do you hear or see the biggest differences? What do you think is the main reason `resample_poly` tends to produce better results?


---

# Part 3: Quantization (ADC view) — SNR, clipping, and oversampling

In this part you will implement a uniform quantizer and measure how quantization noise behaves.

## Task 3.1: Uniform quantization on audio — SNR vs bits


In [ ]:
# Task 3.1 — Implement a uniform quantizer

def uniform_quantize(x, B, xmax):
    """Uniform mid-tread or mid-rise quantizer (your choice), with clipping.

    Inputs:
        x: input signal in roughly [-xmax, xmax]
        B: number of bits (e.g., 2..16)
        xmax: clipping range parameter

    Returns:
        x_hat: quantized signal (float)
        Delta: step size
    """
    # TODO: Implement a B-bit uniform quantizer with clipping range [-xmax, xmax)
    x_hat = # TODO
    Delta = # TODO
    return x_hat, Delta

# Choose a segment (avoid silence if possible)
Fs = fs_audio
x = x_audio.copy()

# Normalize (optional)
x = x / np.max(np.abs(x) + 1e-12)

# Set xmax (you will vary later)
xmax = 1.0

Bs = [2, 4, 6, 8, 12, 16]
snrs = []

for B in Bs:
    x_hat, Delta = uniform_quantize(x, B, xmax)
    e = x_hat - x

    # TODO: Compute SNR in dB
    snr_db = # TODO
    snrs.append(snr_db)

    print(f"B={B}, Delta={Delta:.5f}, SNR={snr_db:.2f} dB")

    # Listen to a couple extreme cases
    if B in [2, 4, 8, 16]:
        display(Audio(x_hat, rate=Fs))

# TODO: Plot SNR (dB) vs B and compare against the theoretical prediction from lecture
# TODO


**Question:**  
1. Does your SNR curve look roughly linear in B? Where does it deviate?  
2. Give one reason your measured SNR might be lower than the “ideal” theory (think: signal distribution, clipping, assumptions behind uniform-noise model).


## Task 3.2: Clipping vs quantization noise tradeoff

Keeping B fixed (e.g., 8 bits), vary `xmax` and observe:
- too small `xmax` → clipping distortion
- too large `xmax` → larger quantization step (more noise)

Use both listening and a simple metric (SNR or MSE).


In [ ]:
# Task 3.2 — Clipping tradeoff

B = 8
x = x_audio / np.max(np.abs(x_audio) + 1e-12)
Fs = fs_audio

xmax_list = [0.3, 0.6, 1.0, 1.5]  # feel free to add

for xmax in xmax_list:
    x_hat, Delta = uniform_quantize(x, B, xmax)
    e = x_hat - x

    # TODO: Compute a quality metric (SNR or MSE)
    metric = # TODO

    print(f"xmax={xmax:.2f}, Delta={Delta:.5f}, metric={metric}")

    # TODO: Plot a zoomed waveform segment showing original vs quantized
    # TODO

    # Optional listening
    display(Audio(x_hat, rate=Fs))


**Question:**  
1. Which `xmax` gave the best overall quality for B=8 in your test?  
2. How can you tell from the waveform/spectrum that clipping occurred?  
3. Explain why making `xmax` larger can *reduce* clipping but *increase* quantization noise.


## Task 3.3: Oversampling + digital anti-aliasing reduces quantization noise

Idea: sample (or represent) the signal at a higher rate, quantize, then digitally low-pass + decimate.

In lecture, oversampling by factor M reduces in-band quantization noise variance by ~M after ideal low-pass + decimation.

In this task you will simulate this digitally:
1. Upsample the signal by M_os (treat as “oversampled representation”)  
2. Quantize at the higher rate with **the same B**  
3. Low-pass + decimate back to the original rate  
4. Compare SNR against direct quantization at the original rate.


In [ ]:
# Task 3.3 — Oversampling experiment

x = x_audio / np.max(np.abs(x_audio) + 1e-12)
Fs = fs_audio

B = 6          # choose a moderate bit-depth where noise is audible
xmax = 1.0

# Baseline: direct quantization at Fs
xq_base, _ = uniform_quantize(x, B, xmax)
e_base = xq_base - x
snr_base = 10*np.log10(np.var(x) / np.var(e_base))
print("Baseline SNR:", snr_base)

M_list = [2, 4, 8, 16]
snr_os = []

for M_os in M_list:
    Fs_os = Fs * M_os

    # TODO: Upsample the signal by M_os to simulate oversampling
    x_os = # TODO

    # Step 2: quantize at high rate
    xq_os, _ = uniform_quantize(x_os, B, xmax)

    # TODO: Design a digital LPF that preserves only the original signal band, then decimate
    numtaps = 401
    cutoff_hz = # TODO
    h = # TODO

    x_lp = signal.lfilter(h, [1.0], xq_os)
    x_back = x_lp[::M_os]

    # Align length
    L = min(len(x_back), len(x))
    x_back = x_back[:L]
    x_ref = x[:L]

    e = x_back - x_ref
    snr = 10*np.log10(np.var(x_ref) / np.var(e))
    snr_os.append(snr)

    print(f"M_os={M_os}, SNR={snr:.2f} dB")

    # Optional: listen to one or two cases
    if M_os in [2, 8, 16]:
        display(Audio(x_back, rate=Fs))

# TODO: Plot SNR vs oversampling factor and compare to the baseline
# TODO


**Question:**  
Does oversampling improve SNR as predicted by theory (~+3 dB per doubling of M)? What role does the **digital low-pass filter + decimation** play in this process? (Hint: connect this to the anti-aliasing concept from Part 1.)


---

# Part 4: Image Compression — Build a JPEG-like pipeline

In Lab 2 you worked in the Fourier domain. Here you will implement the standard compression paradigm:

**Transform → Quantize → Entropy code**

JPEG uses:
- color transform (RGB → YCbCr) + chroma decimation
- 8×8 block DCT
- quantization
- zigzag reorder + run-length encoding (RLE)
- Huffman coding (variable-length codes)

You will implement a simplified version and measure **rate–distortion** tradeoffs (bpp vs PSNR).


## Task 4.1: Color transform + chroma decimation (4:2:0 idea)

JPEG first converts RGB → YCbCr, then **downsamples chroma** channels because human vision is less sensitive to chroma detail.

In this task you will:
1. Convert `img_color` (RGB) to YCbCr  
2. Downsample Cb and Cr by 2× in each dimension  
3. Upsample back and reconstruct RGB  
4. Compare visually and compute an error metric.


In [ ]:
# Task 4.1 — YCbCr and chroma decimation

rgb = util.img_as_float(img_color)

# TODO: Convert RGB to YCbCr color space
ycbcr = # TODO

Y = ycbcr[..., 0]
Cb = ycbcr[..., 1]
Cr = ycbcr[..., 2]

# Visualize channels (rescale for display if needed)
fig, axes = plt.subplots(1, 3, figsize=(12,4))
axes[0].imshow(Y, cmap='gray'); axes[0].set_title("Y (luma)"); axes[0].axis('off')
axes[1].imshow(Cb, cmap='gray'); axes[1].set_title("Cb"); axes[1].axis('off')
axes[2].imshow(Cr, cmap='gray'); axes[2].set_title("Cr"); axes[2].axis('off')
plt.show()

# --- 4:2:0 style chroma decimation ---
H, W = Y.shape
H2, W2 = H//2, W//2

# TODO: Downsample Cb and Cr to half resolution (with anti-aliasing)
Cb_ds = # TODO
Cr_ds = # TODO

# TODO: Upsample back to original resolution
Cb_us = # TODO
Cr_us = # TODO

# Reconstruct YCbCr then RGB
ycbcr_rec = np.stack([Y, Cb_us, Cr_us], axis=-1)

# TODO: Convert YCbCr back to RGB
rgb_rec = # TODO

# Clip for display
rgb_rec = np.clip(rgb_rec, 0, 1)

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(10,4))
axes[0].imshow(rgb); axes[0].set_title("Original RGB"); axes[0].axis('off')
axes[1].imshow(rgb_rec); axes[1].set_title("After chroma decimation"); axes[1].axis('off')
plt.show()

# TODO: Compute reconstruction error (PSNR or MSE)
metric = # TODO
print("Error metric:", metric)


**Question:**  
1. In your output, where (if anywhere) can you visually see damage from chroma decimation?  
2. Why does downsampling chroma usually hurt less than downsampling luma?


## Task 4.2: Block DCT — energy compaction in 8×8 blocks

JPEG uses 8×8 block DCT because many natural image blocks have most energy in low frequencies.

You will:
1. Pick two 8×8 blocks: one “smooth” region, one “edge/texture” region  
2. Compute 2D DCT coefficients for each  
3. Visualize coefficient magnitude and compare energy compaction.


In [ ]:
# Task 4.2 — DCT energy compaction

img = img_gray.copy()  # grayscale float in [0,1]
img_p, orig_shape = pad_to_block(img, block=8)
blocks = block_view(img_p, block=8)  # shape (nBh, nBw, 8, 8)

nBh, nBw, _, _ = blocks.shape
print("Blocks:", blocks.shape)

# TODO: choose two block coordinates (by inspection or trial):
# one smooth, one with edges/texture.
# Hint: try printing / visualizing a few random blocks.
smooth_idx = ( # TODO, # TODO )
edge_idx   = ( # TODO, # TODO )

block_smooth = blocks[smooth_idx[0], smooth_idx[1]]
block_edge   = blocks[edge_idx[0], edge_idx[1]]

# DCT
C_s = dct2(block_smooth)
C_e = dct2(block_edge)

# Visualize blocks and DCT magnitudes
fig, axes = plt.subplots(2, 2, figsize=(8,8))
axes[0,0].imshow(block_smooth, cmap='gray'); axes[0,0].set_title("Smooth block"); axes[0,0].axis('off')
axes[0,1].imshow(np.log1p(np.abs(C_s)), cmap='gray'); axes[0,1].set_title("log|DCT| (smooth)"); axes[0,1].axis('off')
axes[1,0].imshow(block_edge, cmap='gray'); axes[1,0].set_title("Edge block"); axes[1,0].axis('off')
axes[1,1].imshow(np.log1p(np.abs(C_e)), cmap='gray'); axes[1,1].set_title("log|DCT| (edge)"); axes[1,1].axis('off')
plt.show()

# TODO: Quantify energy compaction — what fraction of total energy
# is concentrated in the low-frequency coefficients?
k = 3
def lowfreq_energy_frac(C, k):
    # TODO
    return # TODO

print("Low-freq energy fraction (smooth):", lowfreq_energy_frac(C_s, k))
print("Low-freq energy fraction (edge):", lowfreq_energy_frac(C_e, k))


## Task 4.3: Block DCT + quantization + reconstruction (grayscale)

You will implement a simplified JPEG core on a grayscale image:

1. Convert image to **0–255** range and **center** it (subtract 128)  
2. Split into 8×8 blocks  
3. DCT each block  
4. Quantize with an 8×8 quantization matrix (scaled by a quality parameter)  
5. Dequantize + IDCT + reconstruct image  
6. Measure PSNR and visually inspect artifacts (zoom in!)

**Important:** use the same quant matrix for all blocks.


In [ ]:
# Task 4.3 — JPEG-like core (grayscale)

# Standard-ish JPEG luminance quantization matrix (given)
Q_base = np.array([
    [16, 11, 10, 16, 24, 40, 51, 61],
    [12, 12, 14, 19, 26, 58, 60, 55],
    [14, 13, 16, 24, 40, 57, 69, 56],
    [14, 17, 22, 29, 51, 87, 80, 62],
    [18, 22, 37, 56, 68,109,103, 77],
    [24, 35, 55, 64, 81,104,113, 92],
    [49, 64, 78, 87,103,121,120,101],
    [72, 92, 95, 98,112,100,103, 99]
], dtype=float)

def jpeg_like_encode_gray(img01, q_scale=1.0, block=8):
    """Return quantized DCT coefficient blocks."""
    # TODO: Prepare image for DCT (JPEG operates on centered 0-255 values)
    img = # TODO

    # pad + block view
    img_p, orig_shape = pad_to_block(img, block=block)
    blocks = block_view(img_p, block=block)  # (nBh,nBw,8,8)

    # quant matrix
    Q = Q_base * q_scale

    Cq = np.zeros_like(blocks)
    for i in range(blocks.shape[0]):
        for j in range(blocks.shape[1]):
            C = dct2(blocks[i,j])
            # TODO: Quantize DCT coefficients using Q
            Cq[i,j] = # TODO
    return Cq, orig_shape, Q

def jpeg_like_decode_gray(Cq, orig_shape, Q, block=8):
    """Reconstruct image from quantized coefficients."""
    blocks_rec = np.zeros_like(Cq)
    for i in range(Cq.shape[0]):
        for j in range(Cq.shape[1]):
            # TODO: Dequantize
            C_hat = # TODO
            blocks_rec[i,j] = idct2(C_hat)

    img_rec = unblock_view(blocks_rec)
    H0, W0 = orig_shape
    img_rec = img_rec[:H0, :W0]

    # TODO: Undo the preprocessing from encode
    img_rec01 = # TODO
    return np.clip(img_rec01, 0, 1)

# Sweep a few quality scales (smaller = higher quality, larger = more compression)
q_scales = [0.5, 1.0, 2.0, 4.0]

img01 = img_gray

recons = []
for qs in q_scales:
    Cq, orig_shape, Q = jpeg_like_encode_gray(img01, q_scale=qs)
    img_rec = jpeg_like_decode_gray(Cq, orig_shape, Q)
    recons.append(img_rec)

    p = psnr(img01, img_rec, data_range=1.0)
    print(f"q_scale={qs}: PSNR={p:.2f} dB")

# Display reconstructions (and zoom into a region to see blocking)
fig, axes = plt.subplots(1, len(q_scales)+1, figsize=(4*(len(q_scales)+1),4))
axes[0].imshow(img01, cmap='gray'); axes[0].set_title("Original"); axes[0].axis('off')
for k, qs in enumerate(q_scales):
    axes[k+1].imshow(recons[k], cmap='gray'); axes[k+1].set_title(f"q={qs}"); axes[k+1].axis('off')
plt.show()

# TODO: Choose a crop window that contains edges to inspect blocking artifacts
r0, r1, c0, c1 = # TODO, # TODO, # TODO, # TODO
fig, axes = plt.subplots(1, len(q_scales)+1, figsize=(4*(len(q_scales)+1),4))
axes[0].imshow(img01[r0:r1, c0:c1], cmap='gray'); axes[0].set_title("Orig (zoom)"); axes[0].axis('off')
for k, qs in enumerate(q_scales):
    axes[k+1].imshow(recons[k][r0:r1, c0:c1], cmap='gray'); axes[k+1].set_title(f"q={qs}"); axes[k+1].axis('off')
plt.show()


**Question:**  
Why does the energy compaction property of the DCT (observed in Task 4.2) make it well-suited for image compression? As `q_scale` increases, describe the artifacts you observe in the reconstructed image. Explain why block-based quantization creates discontinuities at block boundaries.


## Task 4.3b: Extend to full-color JPEG pipeline

Now you will extend your grayscale JPEG core to handle **full-color (RGB) images**.

The pipeline:
1. RGB → YCbCr → downsample Cb, Cr by 2× (4:2:0, from Task 4.1)  
2. Apply block DCT + quantization to Y (with **luminance Q matrix**)  
3. Apply block DCT + quantization to Cb, Cr (with **chrominance Q matrix**)  
4. Reconstruct: dequantize + IDCT → upsample Cb/Cr → YCbCr → RGB

The chrominance Q matrix has larger values (coarser quantization) because human vision is less sensitive to chroma detail.


In [ ]:
# Task 4.3b — Full-color JPEG pipeline

# Standard JPEG chrominance quantization matrix (given)
Q_chroma = np.array([
    [17, 18, 24, 47, 99, 99, 99, 99],
    [18, 21, 26, 66, 99, 99, 99, 99],
    [24, 26, 56, 99, 99, 99, 99, 99],
    [47, 66, 99, 99, 99, 99, 99, 99],
    [99, 99, 99, 99, 99, 99, 99, 99],
    [99, 99, 99, 99, 99, 99, 99, 99],
    [99, 99, 99, 99, 99, 99, 99, 99],
    [99, 99, 99, 99, 99, 99, 99, 99]
], dtype=float)

def jpeg_like_encode_color(rgb01, q_scale=1.0, block=8):
    """Encode a color image. Returns dict with Y/Cb/Cr quantized coefficients."""
    # TODO: Convert to YCbCr
    ycbcr = # TODO

    Y  = ycbcr[..., 0]
    Cb = ycbcr[..., 1]
    Cr = ycbcr[..., 2]

    # TODO: Downsample chroma channels by 2x
    H, W = Y.shape
    Cb_ds = # TODO
    Cr_ds = # TODO

    # NOTE: rgb2ycbcr outputs Y in [16,235], Cb/Cr in [16,240].
    # jpeg_like_encode_gray expects [0,1] input, so normalize by /255.
    Cq_Y, shape_Y, Q_Y = jpeg_like_encode_gray(Y / 255.0, q_scale=q_scale, block=block)

    # TODO: Encode Cb and Cr using the chrominance Q matrix (Q_chroma)
    # Remember to normalize the chroma channels the same way.
    Cq_Cb, shape_Cb, Q_Cb = # TODO
    Cq_Cr, shape_Cr, Q_Cr = # TODO

    return {
        "Y": (Cq_Y, shape_Y, Q_Y),
        "Cb": (Cq_Cb, shape_Cb, Q_Cb),
        "Cr": (Cq_Cr, shape_Cr, Q_Cr),
        "orig_shape": (H, W),
    }

def jpeg_like_decode_color(encoded, block=8):
    """Decode back to RGB [0,1]."""
    H, W = encoded["orig_shape"]

    # Decode each channel (returns [0,1] range, need to scale back to YCbCr range)
    Y_rec  = jpeg_like_decode_gray(*encoded["Y"], block=block) * 255.0
    Cb_rec = jpeg_like_decode_gray(*encoded["Cb"], block=block) * 255.0
    Cr_rec = jpeg_like_decode_gray(*encoded["Cr"], block=block) * 255.0

    # TODO: Upsample chroma back to full resolution
    Cb_up = # TODO
    Cr_up = # TODO

    # TODO: Reconstruct RGB from decoded channels
    ycbcr_rec = # TODO
    rgb_rec = # TODO

    return np.clip(rgb_rec, 0, 1)


# --- Test on astronaut ---
rgb01 = util.img_as_float(img_color)

q_scales = [0.5, 1.0, 2.0, 4.0]

fig, axes = plt.subplots(1, len(q_scales)+1, figsize=(4*(len(q_scales)+1), 4))
axes[0].imshow(rgb01); axes[0].set_title("Original"); axes[0].axis('off')

for k, qs in enumerate(q_scales):
    enc = jpeg_like_encode_color(rgb01, q_scale=qs)
    rec = jpeg_like_decode_color(enc)
    p = psnr(rgb01, rec, data_range=1.0)
    print(f"q_scale={qs}: PSNR={p:.2f} dB")
    axes[k+1].imshow(rec); axes[k+1].set_title(f"q={qs}, PSNR={p:.1f}"); axes[k+1].axis('off')

plt.tight_layout()
plt.show()

# Zoom crop
r0, r1, c0, c1 = 100, 200, 200, 350
fig, axes = plt.subplots(1, len(q_scales)+1, figsize=(4*(len(q_scales)+1), 4))
axes[0].imshow(rgb01[r0:r1, c0:c1]); axes[0].set_title("Orig (zoom)"); axes[0].axis('off')
for k, qs in enumerate(q_scales):
    enc = jpeg_like_encode_color(rgb01, q_scale=qs)
    rec = jpeg_like_decode_color(enc)
    axes[k+1].imshow(rec[r0:r1, c0:c1]); axes[k+1].set_title(f"q={qs}"); axes[k+1].axis('off')
plt.tight_layout()
plt.show()


**Question:**  
The chrominance Q matrix has much larger values than the luminance Q matrix (coarser quantization for color channels). Why does this have relatively little impact on perceived quality? Try using the luminance Q matrix for Cb/Cr as well (instead of Q_chroma) and report how PSNR changes. Explain.


## Task 4.4: Zigzag reorder

JPEG reorders 8×8 coefficients so that low frequencies come first and high frequencies come last, making long runs of zeros more likely.

Implement:
- `zigzag_indices(n=8)` → list of (r,c) coordinates
- `zigzag_scan(block)` and inverse `izigzag_scan(vec)`


In [ ]:
# Task 4.4 — Zigzag scan

def zigzag_indices(n=8):
    """Return a list of (r,c) indices in zigzag order for an n×n block."""
    idx = []
    # TODO: implement zigzag ordering
    # Hint: zigzag traverses diagonals with alternating direction.
    return idx

def zigzag_scan(block):
    idx = zigzag_indices(block.shape[0])
    return np.array([block[r, c] for (r, c) in idx])

def izigzag_scan(vec, n=8):
    idx = zigzag_indices(n)
    block = np.zeros((n, n), dtype=vec.dtype)
    for k, (r, c) in enumerate(idx):
        block[r, c] = vec[k]
    return block

# Visualize the zigzag index order as an image of ranks (0..63)
order = np.zeros((8,8), dtype=int)
for k, (r,c) in enumerate(zigzag_indices(8)):
    order[r,c] = k
plt.figure(figsize=(4,4))
plt.imshow(order, cmap='viridis')
plt.colorbar()
plt.title("Zigzag order (rank)")
plt.axis('off')
plt.show()

# Quick sanity check: scan then inverse should recover
test = np.arange(64).reshape(8,8)
v = zigzag_scan(test)
test2 = izigzag_scan(v, n=8)
print("Zigzag invert correct?", np.all(test == test2))


**Question:**  
How does zigzag ordering help RLE produce longer zero-runs? Observe how the run-length distribution changes as `q_scale` increases and explain the trend.


## Task 4.5: Run-length encoding (RLE) of zigzag coefficients

After quantization, many high-frequency coefficients become 0.
RLE compresses long runs of zeros efficiently.

You will:
1. Apply zigzag scan to each block  
2. Encode AC coefficients with RLE tokens like `(run, value)` plus an `EOB` marker  
3. Visualize run-length statistics and how they change with `q_scale`.


In [ ]:
# Task 4.5 — RLE tokens

EOB = ("EOB",)

def rle_ac(vec):
    """RLE encode AC coefficients of a length-64 zigzag vector.
    Return a list of tokens: (run, value) ... EOB
    """
    tokens = []
    # TODO: Encode AC coefficients (vec[1:]) as (run_of_zeros, value) tokens.
    # End the sequence with an EOB marker.
    return tokens

def count_nonzeros(Cq_blocks):
    return np.count_nonzero(Cq_blocks)

img01 = img_gray
q_scales = [0.5, 1.0, 2.0, 4.0]

for qs in q_scales:
    Cq, orig_shape, Q = jpeg_like_encode_gray(img01, q_scale=qs)

    # Build token stream
    tokens_all = []
    runs = []
    for i in range(Cq.shape[0]):
        for j in range(Cq.shape[1]):
            v = zigzag_scan(Cq[i,j].astype(int))
            toks = rle_ac(v)
            tokens_all.extend(toks)
            # collect run lengths
            for tok in toks:
                if tok != EOB:
                    runs.append(tok[0])

    nz = count_nonzeros(Cq)
    total = Cq.size
    print(f"q={qs}: nonzero fraction={nz/total:.4f}, #tokens={len(tokens_all)}")

    # Plot run-length histogram
    plt.figure(figsize=(6,3))
    plt.hist(runs, bins=range(0, 65), density=True)
    plt.title(f"Run-length distribution (q={qs})")
    plt.xlabel("run (# of zeros before a nonzero)")
    plt.ylabel("probability")
    plt.show()


## Task 4.6: Huffman coding (rate estimate)

You will implement Huffman coding to estimate compressed bit-rate.

You are **not** required to pack a real bitstream. Instead, compute:
- Huffman code length for each symbol
- total estimated bits = sum(count(symbol) * code_len(symbol))

Do this for:
1. **Coefficient stream**: all quantized coefficients in zigzag order (including zeros)  
2. **RLE token stream**: tokens from Task 4.5  

Compare resulting **bpp** (bits per pixel).


In [ ]:
# Task 4.6 — Huffman code length estimation

class HuffNode:
    def __init__(self, freq, sym=None, left=None, right=None):
        self.freq = freq
        self.sym = sym
        self.left = left
        self.right = right
    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(symbol_counts):
    """(Given) Build a Huffman tree from symbol frequency counts."""
    if len(symbol_counts) == 0:
        return None
    heap = []
    for sym, freq in symbol_counts.items():
        heapq.heappush(heap, HuffNode(freq, sym=sym))
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = HuffNode(left.freq + right.freq, left=left, right=right)
        heapq.heappush(heap, merged)
    return heap[0] if heap else None

def build_huffman_code_lengths(symbol_counts):
    """Return dict: symbol -> code_length (in bits)."""
    if len(symbol_counts) <= 1:
        return {s: 1 for s in symbol_counts}
    root = build_huffman_tree(symbol_counts)
    code_len = {}
    # TODO: traverse the tree to assign code lengths.
    # Hint: use a stack [(node, depth)]. Start with [(root, 0)].
    # For each node: if it's a leaf (node.sym is not None), record code_len[node.sym] = depth.
    # Otherwise, push left and right children with depth+1.

    return code_len

def estimate_bits(symbol_counts, code_len):
    total = 0
    for s, c in symbol_counts.items():
        total += c * code_len.get(s, 0)
    return total

img01 = img_gray
q_scales = [0.5, 1.0, 2.0, 4.0]

H0, W0 = img01.shape
num_pixels = H0 * W0

for qs in q_scales:
    Cq, orig_shape, Q = jpeg_like_encode_gray(img01, q_scale=qs)

    # --- Stream A: raw coefficients (zigzag order, include all 64) ---
    coeff_stream = []
    for i in range(Cq.shape[0]):
        for j in range(Cq.shape[1]):
            v = zigzag_scan(Cq[i,j].astype(int))
            coeff_stream.extend(list(v))
    countsA = Counter(coeff_stream)

    codeA = build_huffman_code_lengths(countsA)
    bitsA = estimate_bits(countsA, codeA)
    bppA = bitsA / num_pixels

    # --- Stream B: RLE tokens ---
    token_stream = []
    for i in range(Cq.shape[0]):
        for j in range(Cq.shape[1]):
            v = zigzag_scan(Cq[i,j].astype(int))
            token_stream.extend(rle_ac(v))
    countsB = Counter(token_stream)

    codeB = build_huffman_code_lengths(countsB)
    bitsB = estimate_bits(countsB, codeB)
    bppB = bitsB / num_pixels

    print(f"q={qs}: bpp(coeff)={bppA:.3f}, bpp(RLE+Huff)={bppB:.3f}")


**Question:**  
Compare the bpp from RLE+Huffman vs. Huffman-on-raw-coefficients: how much extra compression does RLE as a preprocessing step provide? Explain why.


## Task 4.7: Rate–distortion curves (PSNR vs bpp)

Now you will quantify the tradeoff between:
- **rate** (bits per pixel, bpp)
- **distortion** (PSNR)

Do this for three images:
1. natural image (`camera`)
2. smooth gradient (`gradient`)
3. high-frequency pattern (`checker`)


In [ ]:
# Task 4.7 — Rate–distortion sweep (color)

def rd_point_color(rgb01, q_scale):
    """Return (psnr_db, bpp_est, recon_img) for a color image."""
    enc = jpeg_like_encode_color(rgb01, q_scale=q_scale)
    recon = jpeg_like_decode_color(enc)

    p = psnr(rgb01, recon, data_range=1.0)

    # Rate estimate: sum bits over Y + Cb + Cr channels
    total_bits = 0
    for ch_name in ["Y", "Cb", "Cr"]:
        Cq, _, _ = enc[ch_name]
        token_stream = []
        for i in range(Cq.shape[0]):
            for j in range(Cq.shape[1]):
                v = zigzag_scan(Cq[i,j].astype(int))
                token_stream.extend(rle_ac(v))
        counts = Counter(token_stream)
        if len(counts) > 0:
            code = build_huffman_code_lengths(counts)
            total_bits += estimate_bits(counts, code)

    H0, W0 = rgb01.shape[:2]
    bpp = total_bits / (H0 * W0)
    return p, bpp, recon

images = {
    "astronaut": util.img_as_float(img_color),
    "coffee": util.img_as_float(img_color2),
}

q_scales = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]

rd = {name: {"psnr": [], "bpp": []} for name in images}

for name, img in images.items():
    print("===", name, "===")
    for qs in q_scales:
        p, bpp, _ = rd_point_color(img, qs)
        rd[name]["psnr"].append(p)
        rd[name]["bpp"].append(bpp)
        print(f"q={qs:>4}: PSNR={p:6.2f} dB, bpp={bpp:6.3f}")

# Plot PSNR vs bpp
plt.figure(figsize=(7,5))
for name in images:
    plt.plot(rd[name]["bpp"], rd[name]["psnr"], marker='o', label=name)
plt.xlabel("Estimated rate (bpp)")
plt.ylabel("PSNR (dB)")
plt.title("Rate–Distortion (Color JPEG-like, RLE+Huffman estimate)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## Task 4.8: Compare against real JPEG (Pillow)

Now you are allowed to use Pillow’s JPEG encoder **only for comparison**.

For one image (choose `camera`), do:
1. Save as JPEG in memory (BytesIO) for several `quality` settings  
2. Measure actual file size (bytes → bpp)  
3. Decode back and compute PSNR  
4. Compare the rate–distortion curve to your JPEG-like estimator.


In [ ]:
# Task 4.8 — Pillow JPEG comparison (color)

rgb01 = util.img_as_float(img_color)
img_u8 = (np.clip(rgb01, 0, 1) * 255).round().astype(np.uint8)

qualities = [10, 20, 30, 50, 70, 90]
psnr_list = []
bpp_list = []

H0, W0 = img_u8.shape[:2]

for q in qualities:
    buf = BytesIO()
    Image.fromarray(img_u8).save(buf, format='JPEG', quality=q)
    jpeg_bytes = buf.getvalue()
    size_bytes = len(jpeg_bytes)

    buf2 = BytesIO(jpeg_bytes)
    img_rec_u8 = np.array(Image.open(buf2))
    img_rec01 = img_rec_u8.astype(np.float32) / 255.0

    p = psnr(rgb01, img_rec01, data_range=1.0)
    bpp = (8 * size_bytes) / (H0 * W0)

    psnr_list.append(p)
    bpp_list.append(bpp)

    print(f"JPEG quality={q:>3}, size={size_bytes/1024:.1f} KB, bpp={bpp:.3f}, PSNR={p:.2f} dB")

# Plot against your estimator curve
plt.figure(figsize=(7,5))
plt.plot(bpp_list, psnr_list, marker='s', label='Pillow JPEG (real)')
if "astronaut" in rd:
    plt.plot(rd["astronaut"]["bpp"], rd["astronaut"]["psnr"], marker='o', label='Your JPEG-like (astronaut)')
plt.xlabel("Rate (bpp)")
plt.ylabel("PSNR (dB)")
plt.title("Real JPEG vs Your JPEG-like (color astronaut)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


**Question:**  
Compare the two R-D curves for astronaut and coffee: which image is more compressible and why? Now compare your pipeline against Pillow JPEG on the same astronaut image. Give **two reasons** why real JPEG performs differently (think: coding details / headers / DC prediction / optimized Huffman tables).


---

# Part 5: Design tradeoff — Downsample then compress vs compress only

Now you will combine ideas from sampling + compression.

Goal: Under the same size budget, compare two strategies:

- **Strategy A (sampling-first):** anti-aliased downsample → compress at higher quality → decode → upsample for viewing  
- **Strategy B (compression-first):** compress the full-resolution image more aggressively → decode

You must choose parameters so both strategies meet the *same* total bit budget (within a small tolerance).


## Task 5.1: Same-budget comparison

Use `camera` image. Pick a total budget (in bits or KB), then find parameters for both strategies.

Suggested: budget = **0.5 bpp relative to the original resolution** (you may choose a different budget, but report it clearly).


In [ ]:
# Task 5.1 — Same-budget comparison (color)

rgb01 = util.img_as_float(img_color)
H0, W0 = rgb01.shape[:2]
budget_bpp = 0.5
budget_bits = budget_bpp * H0 * W0
print("Budget bits:", budget_bits)

q_candidates = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]

# --- Strategy B: compress full-res directly ---
best_B = None
for qs in q_candidates:
    p, bpp, rec = rd_point_color(rgb01, qs)
    bits = bpp * H0 * W0
    if bits <= budget_bits:
        best_B = (qs, p, bpp, rec, bits)
        break
print("Best Strategy B:", (best_B[0], f"PSNR={best_B[1]:.2f}", f"bpp={best_B[2]:.3f}") if best_B else None)

# --- Strategy A: downsample then compress ---
scale = 0.5
# TODO: Downsample the image with anti-aliasing
img_ds = # TODO

H1, W1 = img_ds.shape[:2]

best_A = None
for qs in q_candidates:
    p_ds, bpp_ds, rec_ds = rd_point_color(img_ds, qs)
    bits = bpp_ds * H1 * W1
    if bits <= budget_bits:
        best_A = (qs, p_ds, bpp_ds, rec_ds, bits)
        break
print("Best Strategy A (on low-res):", (best_A[0], f"PSNR_lr={best_A[1]:.2f}", f"bpp_lr={best_A[2]:.3f}") if best_A else None)

# TODO: Upsample the reconstruction back to original resolution for comparison
rec_A_up = # TODO
rec_B = best_B[3]

# Compare PSNR vs original full-res
pA = psnr(rgb01, rec_A_up, data_range=1.0)
pB = psnr(rgb01, rec_B, data_range=1.0)
print(f"PSNR vs original: Strategy A(up) = {pA:.2f}, Strategy B = {pB:.2f}")

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15,5))
axes[0].imshow(rgb01); axes[0].set_title("Original"); axes[0].axis('off')
axes[1].imshow(rec_A_up); axes[1].set_title(f"A: ds+compress (PSNR={pA:.1f})"); axes[1].axis('off')
axes[2].imshow(rec_B); axes[2].set_title(f"B: compress only (PSNR={pB:.1f})"); axes[2].axis('off')
plt.tight_layout()
plt.show()

# Zoom crop
r0, r1, c0, c1 = 100, 220, 180, 360
fig, axes = plt.subplots(1, 3, figsize=(15,5))
axes[0].imshow(rgb01[r0:r1, c0:c1]); axes[0].set_title("Orig (zoom)"); axes[0].axis('off')
axes[1].imshow(rec_A_up[r0:r1, c0:c1]); axes[1].set_title("A (zoom)"); axes[1].axis('off')
axes[2].imshow(rec_B[r0:r1, c0:c1]); axes[2].set_title("B (zoom)"); axes[2].axis('off')
plt.tight_layout()
plt.show()


**Question:**  
Under the same bit budget, which strategy looks better to you? Describe the artifacts in terms of blur vs. blocking vs. ringing. Does PSNR match your subjective preference? Why might PSNR disagree with what you see?


---

# Final reflection

Answer in 5–8 sentences total:

- What is the conceptual link between **ADC quantization** (Part 3) and **lossy compression quantization** (Part 4)?  
- In your own words, why does the "Transform → Quantize → Entropy-code" pipeline work so well?  
- Where did your experiments deviate from ideal theory, and what do you think caused the gap?
